# Session 2 — Data Download and Verification
Run after executing src/data/download.py

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path(".").resolve().parent
RAW = ROOT / "data" / "raw"
print("Root:", ROOT)

## 1. Cell Model Passports

In [ ]:
mapping = pd.read_csv(RAW / "mapping" / "model_list_latest.csv.gz")
print("Shape:", mapping.shape)
print("Columns:", mapping.columns.tolist())
mapping.head(3)

## 2. GDSC2

In [ ]:
gdsc = pd.read_csv(RAW / "gdsc2" / "GDSC2_fitted_dose_response.csv")
print("Shape:", gdsc.shape)
print("Cell lines:", gdsc["CELL_LINE_NAME"].nunique())
print("Drugs:", gdsc["DRUG_NAME"].nunique())
gdsc.head(3)

## 3. CCLE RNA-seq

In [ ]:
rna = pd.read_csv(RAW / "ccle" / "OmicsExpressionProteinCodingGenesTPMLogp1.csv", index_col=0)
print("Shape:", rna.shape)  # expect (~1019, ~19000)
print("Index sample:", rna.index[:5].tolist())  # expect ACH-XXXXXX format
rna.iloc[:3, :5]

## 4. ProCan Proteomics

In [ ]:
prot = pd.read_csv(RAW / "procan" / "procan_protein_matrix.csv", index_col=0)
print("Shape:", prot.shape)  # expect (~949, ~8498)
print("Index sample:", prot.index[:5].tolist())  # expect SIDM format
print("Missing values (%):", prot.isna().mean().mean() * 100)
prot.iloc[:3, :5]

## 5. ChEMBL API Test

In [ ]:
from chembl_webresource_client.new_client import new_client
molecule = new_client.molecule

test_drugs = ["Erlotinib", "Imatinib", "Gefitinib", "Paclitaxel", "Docetaxel"]
for drug in test_drugs:
    hits = molecule.filter(pref_name__iexact=drug).only(["molecule_chembl_id", "molecule_structures"])
    smiles = None
    for r in hits:
        if r.get("molecule_structures"):
            smiles = r["molecule_structures"].get("canonical_smiles")
            break
    print(f"{drug}: {smiles[:50] if smiles else 'NOT FOUND'}")